In [29]:
#Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
#Download Data
data = pd.read_excel("../data/ctech projects (CA).xlsx")

print("Shape:", data.shape)

Shape: (46608, 30)


In [3]:
#Create Modeling Dataset
model_data = data.copy()

In [4]:
#The Stats of this specific columnn
model_data["Actual Eng Hours"].describe()


count    45924.000000
mean        12.594231
std         19.923312
min          0.000000
25%          3.250000
50%          7.500000
75%         15.042500
max        971.650000
Name: Actual Eng Hours, dtype: float64

In [5]:
#Total rows with missing hours
print(
    model_data["Actual Eng Hours"].isna().sum()
)

684


In [6]:
#How many missing values does each column still have?
model_data.isna().sum().sort_values(ascending=False)

Product Type Segment            45377
Flex Standards                  33323
Avg Scoped Hours Lab            19239
Product Group                   16853
CCN                             12671
Avg Scoped Hours Engineer        3328
Actual Lab Hours                  684
Actual Eng Hours                  684
Service Program                   576
Service Detail                    544
Product Type                      335
Service Catalog Segment             2
Service Catalog Item Number         2
Service Catalog Category            2
Service Catalog Sub Category        2
Industry Name                       0
Industry Code                       0
Service Line Code                   0
Confirmation Year                   0
Flex Project UUID                   0
Handler Region                      0
Product Type Code                   0
Service Line Name                   0
Ship to Account Number              0
Ship to Customer Region             0
Has Test Task Flag                  0
Flex Project

In [7]:
#1. Drop rows where the ENG target variable is missing
model_data = model_data.dropna( subset =["Actual Eng Hours"])

In [8]:
#2. Create log-transformed target
model_data["Log_Eng_Hours"] = np.log1p(model_data["Actual Eng Hours"])

In [9]:
model_data["Product Type Segment"].isna().mean()*100

np.float64(97.66570856197195)

In [10]:
#Drop obvious unneccessary /leakage columns
drop_cols = [
    "Flex Project UUID",
    "Actual Eng Hours",
    "Actual Lab Hours",
    "Total Scoped Hours",
    "Product Type Segment",
    "Avg Scoped Hours Lab",
    "Avg Scoped Hours Engineer"
    #"Ship to Account Number",
]

model_data = model_data.drop(columns=drop_cols, errors="ignore")

In [11]:
cat_cols = model_data.select_dtypes(
    include=["object"]
).columns

for col in cat_cols:
    model_data[col] = model_data[col].fillna("UNKNOWN")



C:\Users\105802\AppData\Local\Temp\ipykernel_27652\2678929310.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = model_data.select_dtypes(


In [12]:
print(model_data.shape)

for col in model_data.select_dtypes(include=["object"]).columns:
    print(
        col,
        model_data[col].nunique()
    )


(45924, 24)
Confirmation Year 3
Handler Region 3
Product Group 218
Product Type 771
Product Type Code 10
Industry Name 31
Industry Code 31
Service Line Code 167
Service Line Name 167
Service Detail 2426
Service Program 540
Service Catalog Category 73
Service Catalog Item Number 2554
Service Catalog Segment 22
Service Catalog Sub Category 348
CCN 209
Ship to Customer Region 3
Has Test Task Flag 2
Flex Standards 3438


C:\Users\105802\AppData\Local\Temp\ipykernel_27652\1256315231.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in model_data.select_dtypes(include=["object"]).columns:


In [13]:
model_data["Flex Standards"].value_counts().head(20)

Flex Standards
UNKNOWN                                                                                                                                                                                                                                                                                                                                                                                                                                                           32712
IEC 62368-1:2018                                                                                                                                                                                                                                                                                                                                                                                                                                                    709
IEC 62133-2:2017, IEC 62133-2:2017/AMD1:2021                             

In [14]:
top20 = model_data["Flex Standards"].value_counts().head(20)

print(top20)
print()
print("Percent covered by top 20:",
      top20.sum() / len(model_data) * 100)


Flex Standards
UNKNOWN                                                                                                                                                                                                                                                                                                                                                                                                                                                           32712
IEC 62368-1:2018                                                                                                                                                                                                                                                                                                                                                                                                                                                    709
IEC 62133-2:2017, IEC 62133-2:2017/AMD1:2021                             

In [15]:
top20 = model_data["Service Catalog Item Number"].value_counts().head(20)

print(top20)
print()
print("Percent covered by top 20:",
      top20.sum() / len(model_data) * 100)

Service Catalog Item Number
30166970-CPQ    1647
30054096-CPQ    1570
30056244-CPQ    1486
30054083-CPQ    1318
30054234-CPQ    1288
30056245-CPQ    1238
30056282-CPQ    1198
30817083-CPQ    1129
30056235-CPQ    1020
30056268-CPQ     964
30166969-CPQ     961
29258136-CPQ     862
30055065-CPQ     827
30056234-CPQ     715
30055066-CPQ     715
30055064-CPQ     650
30054242-CPQ     575
30053638-CPQ     546
29258124-CPQ     496
30150000-CPQ     474
Name: count, dtype: int64

Percent covered by top 20: 42.85123247103911


In [16]:
vc = model_data["Service Catalog Item Number"].value_counts()

print("Unique values:", len(vc))
print("Appear once:", (vc == 1).sum())
print("Percent appearing once:",
      (vc == 1).sum() / len(vc) * 100)


Unique values: 2554
Appear once: 1351
Percent appearing once: 52.8974158183242


In [17]:
high_card_cols = [
    "Service Catalog Item Number",
    "Flex Standards",
    "Service Detail",
    "Product Type",
    "Service Program"
]

for col in high_card_cols:
    vc = model_data[col].value_counts()
    common_values = vc[vc >= 10].index
    
    model_data[col] = np.where(
        model_data[col].isin(common_values),
        model_data[col],
        "OTHER"
    )


In [18]:
#Double check missing values again
model_data.isna().sum().sort_values(ascending=False)

Confirmation Year               0
Handler Region                  0
Product Group                   0
Product Type                    0
Product Type Code               0
Industry Name                   0
Industry Code                   0
Service Line Code               0
Service Line Name               0
Service Detail                  0
Service Program                 0
Service Catalog Category        0
Service Catalog Item Number     0
Service Catalog Segment         0
Service Catalog Sub Category    0
CCN                             0
Ship to Customer Region         0
Has Test Task Flag              0
Ship to Account Number          0
Flex Standards                  0
Standard Count                  0
Flex Project Count              0
Test Count                      0
Log_Eng_Hours                   0
dtype: int64

In [19]:
#List all the columns we have now after cleaning
model_data.columns.tolist()

['Confirmation Year',
 'Handler Region',
 'Product Group',
 'Product Type',
 'Product Type Code',
 'Industry Name',
 'Industry Code',
 'Service Line Code',
 'Service Line Name',
 'Service Detail',
 'Service Program',
 'Service Catalog Category',
 'Service Catalog Item Number',
 'Service Catalog Segment',
 'Service Catalog Sub Category',
 'CCN',
 'Ship to Customer Region',
 'Has Test Task Flag',
 'Ship to Account Number',
 'Flex Standards',
 'Standard Count',
 'Flex Project Count',
 'Test Count',
 'Log_Eng_Hours']

In [20]:
# Features
X = model_data.drop(columns=["Log_Eng_Hours"])

# Target
y = model_data["Log_Eng_Hours"]

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (45924, 23)
y shape: (45924,)


In [21]:
# X_encoded = pd.get_dummies(X, drop_first=True)

# print("Encoded X shape:", X_encoded.shape)


In [22]:
# X_train, X_test, y_train, y_test = train_test_split(
#     X_encoded,
#     y,
#     test_size=0.2,
#     random_state=42
# )

# print("X_train:", X_train.shape)
# print("X_test:", X_test.shape)
# print("y_train:", y_train.shape)
# print("y_test:", y_test.shape)


In [23]:
# #find all text/categorical columns that need to be converted into numeric ML features
# cat_cols = X.select_dtypes(include=["object"]).columns

# print(cat_cols)
# print("Total categorical columns:", len(cat_cols))


In [24]:
# #Convert all categorical/text columns into numeric columns that the ML model can understand
# X_encoded = pd.get_dummies(
#     X,
#     columns=cat_cols,
#     drop_first=True
# )

# print(X_encoded.shape)


In [25]:
# #Double check that all columns are now numeric data types for machine learning
# X_encoded.dtypes.value_counts()

In [26]:
# #Verify that no text/oject columns are left after encoding
# X_encoded.select_dtypes(include=["object"]).columns

In [27]:
# X = X_encoded
# y = model_data["Log_Eng_Hours"]

In [28]:
# Save preprocessed modeling data
model_data.to_pickle("../data/preprocessed_model_data.pkl")

print("Saved preprocessed data:")
print(model_data.shape)

Saved preprocessed data:
(45924, 24)
